In [2]:
pip install pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("exercise.csv")
df.head()

,Unnamed: 0,Title,Desc,Type,BodyPart,Equipment,Level,Rating,RatingDesc
0,0,Partner plank band row,The partner plank band row is an abdominal exe...,Strength,Abdominals,Bands,Intermediate,0.0,NaN
1,1,Banded crunch isometric hold,The banded crunch isometric hold is an exercis...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
2,2,FYR Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
3,3,Banded crunch,The banded crunch is an exercise targeting the...,Strength,Abdominals,Bands,Intermediate,NaN,NaN
4,4,Crunch,The crunch is a popular core exercise targetin...,Strength,Abdominals,Bands,Intermediate,NaN,NaN


In [5]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2918 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2918 non-null   int64  
 1   Title       2918 non-null   object 
 2   Desc        1368 non-null   object 
 3   Type        2918 non-null   object 
 4   BodyPart    2918 non-null   object 
 5   Equipment   2886 non-null   object 
 6   Level       2918 non-null   object 
 7   Rating      1031 non-null   float64
 8   RatingDesc  862 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 205.3+ KB


Unnamed: 0       0
Title            0
Desc          1550
Type             0
BodyPart         0
Equipment       32
Level            0
Rating        1887
RatingDesc    2056
dtype: int64

In [7]:
df.rename(columns={
    "Title": "name",
    "Desc": "description",
    "Type": "category",
    "BodyPart": "muscle",
    "Equipment": "equipment",
    "Level": "difficulty",
    "Rating": "rating",
    "RatingDesc": "rating_description"
}, inplace=True)

In [8]:
df["name"] = df["name"].str.strip().str.title()
df["muscle"] = df["muscle"].str.strip().str.title()
df["equipment"] = df["equipment"].str.strip().str.title()
df["difficulty"] = df["difficulty"].str.strip().str.title()

In [9]:
df["description"] = df["description"].fillna("No description available")
df["rating"] = df["rating"].fillna(np.nan)

In [10]:
df.drop_duplicates(subset=["name"], inplace=True)

In [11]:
muscle_map = {
    "Abs": "Core",
    "Abdominals": "Core",
    "Glutes": "Glutes",
    "Hamstrings": "Legs",
    "Quadriceps": "Legs",
    "Back": "Back",
    "Chest": "Chest",
    "Shoulders": "Shoulders",
    "Biceps": "Arms",
    "Triceps": "Arms"
}

df["muscle"] = df["muscle"].replace(muscle_map)

In [12]:
equipment_map = {
    "Body Only": "Bodyweight",
    "Machine": "Machine",
    "Dumbbell": "Dumbbell",
    "Barbell": "Barbell",
    "Kettlebells": "Kettlebell",
    "Cable": "Cable"
}

df["equipment"] = df["equipment"].replace(equipment_map)

In [13]:
df = df[df["name"].notna()]
df = df[df["muscle"].notna()]
df = df[df["equipment"].notna()]

In [14]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2877 entries, 0 to 2917
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          2877 non-null   int64  
 1   name                2877 non-null   object 
 2   description         2877 non-null   object 
 3   category            2877 non-null   object 
 4   muscle              2877 non-null   object 
 5   equipment           2877 non-null   object 
 6   difficulty          2877 non-null   object 
 7   rating              996 non-null    float64
 8   rating_description  828 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 224.8+ KB


In [15]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [16]:
df["rating"] = df["rating"].fillna(df["rating"].mean())

In [17]:
df.to_csv("clean_exercises.csv", index=False)

In [18]:
import pandas as pd

df = pd.read_csv("clean_exercises.csv")

df.head()

,name,description,category,muscle,equipment,difficulty,rating,rating_description
0,Partner Plank Band Row,The partner plank band row is an abdominal exe...,Strength,Core,Bands,Intermediate,0.000000,NaN
1,Banded Crunch Isometric Hold,The banded crunch isometric hold is an exercis...,Strength,Core,Bands,Intermediate,5.933635,NaN
2,Fyr Banded Plank Jack,The banded plank jack is a variation on the pl...,Strength,Core,Bands,Intermediate,5.933635,NaN
3,Banded Crunch,The banded crunch is an exercise targeting the...,Strength,Core,Bands,Intermediate,5.933635,NaN
4,Crunch,The crunch is a popular core exercise targetin...,Strength,Core,Bands,Intermediate,5.933635,NaN


In [19]:
df["name"] = df["name"].str.lower()
df["muscle"] = df["muscle"].str.lower()
df["category"] = df["category"].str.lower()

In [20]:
def classify_glutes(name, muscle):

    name = name.lower()
    muscle = muscle.lower()

    # =========================
    # 🍑 GLUTE MAXIMUS (power + compound movements)
    # =========================
    if any(x in name for x in [
        "squat", "hip thrust", "thrust", "deadlift",
        "romanian deadlift", "russian deadlift", "sumo deadlift",
        "glute bridge", "bridge", "frog pump", "frog pump glute",
        "hip lift", "kas glute bridge",
        "lunge", "split squat", "bulgarian split squat",
        "reverse lunge", "step up", "step-up",
        "deficit deadlift",
        "back extension", "hyperextension",
        "good morning", "kettlebell swing", "swing"
    ]):
        return "glute_maximus"

    # =========================
    # 🍑 GLUTE MEDIUS (side glute / shape / abduction)
    # =========================
    if any(x in name for x in [
        "abduction", "hip abduction", "banded abduction",
        "lateral walk", "band walk", "monster walk",
        "side lying", "side leg raise",
        "clamshell", "clam",
        "fire hydrant",
        "kickback", "glute kickback", "cable kickback",
        "donkey kick", "standing kickback",
        "lateral band", "banded lateral",
        "curtsy lunge",
        "lateral lunge", "side lunge",
        "side step"
    ]):
        return "glute_medius"

    # =========================
    # 🍑 GLUTE MINIMUS (stability + deep control + single leg)
    # =========================
    if any(x in name for x in [
        "single leg", "single-leg",
        "unilateral", "one leg",
        "pistol squat", "pistol",
        "single leg bridge",
        "single leg squat",
        "single leg deadlift",
        "balance",
        "step down", "step-down",
        "stability",
        "hip hike",
        "single leg stance",
        "offset"
    ]):
        return "glute_minimus"

    # =========================
    # 🍑 GENERAL GLUTES (fallback when muscle column confirms glutes)
    # =========================
    if "glute" in muscle:
        return "glute_general"

    return None

In [29]:
df["muscle_region"] = df.apply(
    lambda row: classify_glutes(row["name"], row["muscle"]),
    axis=1
)

In [21]:
def classify_abs(name, muscle):

    name = name.lower()
    muscle = muscle.lower()

    # =========================
    # 🔥 RECTUS ABDOMINIS (six-pack / flexion movement)
    # =========================
    if any(x in name for x in [
        "crunch", "sit up", "sit-up", "situp",
        "ab crunch", "weighted crunch", "decline crunch", "cable crunch",
        "jackknife", "v-up", "v up", "v-sit", "v sit",
        "bicycle crunch", "cross crunch",
        "reverse crunch",
        "leg raise", "leg raises",
        "knee raise",
        "toe touch", "toe to bar",
        "flutter kick", "scissor kick",
        "hollow hold", "hollow body"
    ]):
        return "rectus_abdominis"

    # =========================
    # 🔥 OBLIQUES (rotation + side flexion)
    # =========================
    if any(x in name for x in [
        "twist", "russian twist",
        "woodchopper", "wood chopper",
        "side bend",
        "side crunch",
        "oblique",
        "windshield wiper", "floor wiper",
        "pallof press",
        "anti-rotation",
        "lateral flexion",
        "bicycle crunch"  # overlaps but obliques heavy involvement
    ]):
        return "obliques"

    # =========================
    # 🔥 TRANSVERSE ABDOMINIS / DEEP CORE (stability + bracing)
    # =========================
    if any(x in name for x in [
        "plank", "forearm plank", "side plank",
        "mountain climber", "mountain climbers",
        "dead bug",
        "bird dog",
        "ab rollout", "ab wheel", "rollout", "roll out",
        "stability",
        "vacuum", "stomach vacuum",
        "transverse",
        "anti-extension",
        "hanging leg raise", "captain's chair",
        "roman chair",
        "dragon flag"
    ]):
        return "transverse_core"

    # =========================
    # 🔥 GENERAL ABS (fallback for unclear abdominal tagging)
    # =========================
    if "ab" in muscle or "abdominal" in muscle or "core" in muscle:
        return "abs_general"

    return None

In [30]:
df["muscle_region"] = df.apply(
    lambda row: classify_abs(row["name"], row["muscle"]),
    axis=1
)

In [22]:
def classify_back(name, muscle):
   
    name = name.lower()  # case-insensitive
    muscle = muscle.lower()  # case-insensitive

    # TRAPS (trapezius — elevation, retraction, shrugs, upright pulls, face pulls often hit upper/mid)
    if any(x in name for x in [
        "shrug", "trap shrug", "barbell shrug", "dumbbell shrug", "upright row", "farmer walk",
        "face pull", "y raise", "rear delt fly"  # upper traps/rear emphasis
        "power shrug", "overhead shrug", "high pull", "clean pull"
    ]):
        return "back_traps"
    
    # RHOMBOIDS (mid-back retraction, scapular squeeze — horizontal rows, pull-aparts, band pulls)
    if any(x in name for x in [
        "row", "seated row", "bent over row", "barbell row", "dumbbell row", "cable row",
        "inverted row", "body row", "t-bar row", "meadows row", "landmine row",
        "band pull apart", "pull apart", "rear delt", "scapular retraction", "prone cobra",
        "iyt raise", "w raise", "scap pull"
    ]):
        return "back_rhomboids"
    
    # LATS (latissimus dorsi — vertical pulls for width, straight-arm pulls, pulldowns)
    if any(x in name for x in [
        "pull up", "pull-up", "chin up", "chin-up", "lat pulldown", "lat pull", "pulldown",
        "wide grip", "straight arm pulldown", "straight-arm", "pullover", "dumbell pullover",
        "deadlift"  # conventional/sumo often hits lats hard, but secondary
    ]):
        return "back_lats"
    
    # LOWER BACK / ERECTORS fallback (if extension-focused, not covered above)
    if any(x in name for x in [
        "back extension", "hyperextension", "good morning", "superman", "stiff leg deadlift"
    ]):
        return "back_lower_erectors"
    
    # General catch-all if BodyPart indicates back/lats/traps but no specific match
    if any(x in muscle.lower() for x in ["back", "lat", "trap", "rhomboid"]):
        return "back_general"
    
    return None

In [31]:
df["muscle_region"] = df.apply(
    lambda row: classify_back(row["name"], row["muscle"]),
    axis=1
)

In [23]:
def classify_arms(name, muscle):
    name = name.lower()
    muscle = muscle.lower()
    # BICEPS (elbow flexion, curls)
    if any(x in name for x in [
        "curl", "bicep curl", "barbell curl", "dumbbell curl", "hammer curl", "preacher curl",
        "concentration curl", "cable curl", "ez bar curl", "chin up", "chin-up"  # secondary
    ]):
        return "arms_biceps"
    
    # TRICEPS (elbow extension, pushdowns, extensions)
    if any(x in name for x in [
        "tricep", "triceps extension", "overhead extension", "skull crusher", "french press",
        "pushdown", "tricep pushdown", "dip", "dips", "close grip bench", "diamond push up"
    ]):
        return "arms_triceps"
    
    # FOREARMS / GRIP (wrist, grip-focused)
    if any(x in name for x in [
        "wrist curl", "reverse curl", "wrist extension", "farmer walk", "grip", "forearm"
    ]):
        return "arms_forearms"
    
    if "arm" in muscle.lower() or "bicep" in muscle.lower() or "tricep" in muscle.lower():
        return "arms_general"
    
    return None

In [32]:
df["muscle_region"] = df.apply(
    lambda row: classify_arms(row["name"], row["muscle"]),
    axis=1
)

In [24]:
def classify_legs(name, muscle):
   
    name = name.lower()  # case-insensitive matching
    muscle = muscle.lower()
    # ABDUCTORS (hip abduction — outer hip/leg away from body, side moves, often glute med/min overlap)
    if any(x in name for x in [
        "abduction", "hip abduction", "side abduction", "banded abduction", "machine abduction",
        "thigh abductor", "standing abduction", "cable abduction", "lateral walk", "band walk",
        "monster walk", "side step", "lateral band", "side lying abduction", "clamshell"  # abduction emphasis
    ]):
        return "legs_abductors"
    
    # HIP FLEXORS (hip flexion — knee drive toward chest, raises, marches, often anterior chain)
    if any(x in name for x in [
        "leg raise", "leg raises", "hanging leg raise", "captain's chair", "knee raise",
        "mountain climber", "mountain climbers", "high knee", "hip flexor march", "banded march",
        "straight leg raise", "flutter kick", "scissor kick", "psoas sit-up", "kettlebell knee drive",
        "groiners", "lunge"  # dynamic flexion in lead leg
    ]):
        return "legs_hip_flexors"
    
    # QUADRICEPS (knee extension dominant, front thigh — squats, presses, extensions)
    if any(x in name for x in [
        "squat", "front squat", "goblet squat", "leg press", "hack squat", "bulgarian split",
        "split squat", "lunge", "reverse lunge", "step up", "leg extension", "sissy squat",
        "wall sit", "thigh"  # general thigh moves often quad-heavy
    ]):
        return "legs_quads"
    
    # HAMSTRINGS (knee flexion/hip extension dominant, posterior chain)
    if any(x in name for x in [
        "deadlift", "romanian deadlift", "stiff leg deadlift", "good morning", "leg curl",
        "lying leg curl", "seated leg curl", "hamstring curl", "nordic curl", "glute ham raise"
    ]):
        return "legs_hamstrings"
    
    # General catch-all if BodyPart indicates legs/quads/hams/abductors but no specific match
    if any(x in muscle.lower() for x in ["leg", "quad", "hamstring", "abductor", "hip flexor", "thigh"]):
        return "legs_general"
    
    return None

In [33]:
df["muscle_region"] = df.apply(
    lambda row: classify_legs(row["name"], row["muscle"]),
    axis=1
)

In [26]:
def classify_chest(name, muscle):
    name = name.lower()
    
    # UPPER CHEST (incline focus)
    if any(x in name for x in [
        "incline", "upper chest", "incline bench", "incline dumbbell", "incline press"
    ]):
        return "chest_upper"
    
    # LOWER CHEST (decline, dips)
    if any(x in name for x in [
        "decline", "lower chest", "decline bench", "dip", "dips", "chest dip"
    ]):
        return "chest_lower"
    
    # GENERAL / MID CHEST (flat bench, flyes, push ups)
    if any(x in name for x in [
        "bench press", "barbell bench", "dumbbell bench", "chest press", "push up", "push-up",
        "fly", "pec fly", "cable fly", "pec deck", "chest fly"
    ]):
        return "chest_pectorals"
    
    if "chest" in muscle.lower() or "pec" in muscle.lower():
        return "chest_general"
    
    return None

In [34]:
df["muscle_region"] = df.apply(
    lambda row: classify_chest(row["name"], row["muscle"]),
    axis=1
)

In [27]:
def classify_calves(name, muscle):
    name = name.lower()
    
    # STANDING / GASTROCNEMIUS (knee straight)
    if any(x in name for x in [
        "standing calf raise", "calf raise", "standing raise", "donkey calf", "smith calf raise"
    ]):
        return "calves_gastrocnemius"
    
    # SEATED / SOLEUS (knee bent)
    if any(x in name for x in [
        "seated calf raise", "seated raise", "soleus", "leg press calf"
    ]):
        return "calves_soleus"
    
    # GENERAL / OTHER (jumps, raises)
    if any(x in name for x in [
        "calf", "jump rope", "toe raise", "heel drop", "eccentric calf"
    ]):
        return "calves_general"
    
    if "calf" in muscle.lower() or "calve" in muscle.lower():
        return "calves_general"
    
    return None

In [35]:
df["muscle_region"] = df.apply(
    lambda row: classify_calves(row["name"], row["muscle"]),
    axis=1
)

In [36]:
df["rating"] = df["rating"].round(1)

In [37]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce").round(1)

In [38]:
df["rating"] = df["rating"].fillna(df["rating"].mean()).round(1)

In [39]:
df["muscle_region"].value_counts()

muscle_region
calves_gastrocnemius    27
calves_general          23
Name: count, dtype: int64

In [49]:
def classify_adductors(name, muscle):
    name = name.lower()

    if any(x in name for x in [
        "adductor", "adduction", "groin", "inner thigh"
    ]):
        return "legs_adductors"

    return None

In [50]:
def classify_plyo(name):
    name = name.lower()

    if any(x in name for x in [
        "jump", "hop", "box jump", "cone", "agility", "carioca"
    ]):
        return "plyometric"

    return None

In [51]:
def classify_mobility(name):
    name = name.lower()

    if any(x in name for x in [
        "stretch", "foam roll", "smr"
    ]):
        return "mobility"

    return None

In [59]:
def classify_shoulders(name, muscle):
    name = name.lower()
    
    # ANTERIOR / FRONT DELTS (overhead pressing dominant)
    if any(x in name for x in [
        "press", "shoulder press", "military press", "overhead press", "barbell press", "kettlebell press",
        "clean and press", "clean and jerk", "snatch", "landmine press", "bradford press"
    ]):
        return "shoulders_anterior"
    
    # LATERAL / SIDE DELTS (raises)
    if any(x in name for x in [
        "lateral raise", "side raise", "band lateral raise"
    ]):
        return "shoulders_lateral"
    
    # REAR / POSTERIOR DELTS + SCAPULAR (pull-aparts, rear-focused)
    if any(x in name for x in [
        "pull-apart", "band pull-apart", "rear delt", "thread the needle", "over-and-back"
    ]):
        return "shoulders_rear"
    
    # ROTATOR CUFF / STABILITY (rotations)
    if any(x in name for x in [
        "internal rotation", "external rotation"
    ]):
        return "shoulders_rotator_cuff"
    
    if "shoulder" in muscle.lower() or "deltoid" in muscle.lower():
        return "shoulders_general"
    
    return None


def classify_neck(name, muscle):
    name = name.lower()
    
    if any(x in name for x in ["extension", "face down", "prone", "back"]):
        return "neck_extension"  # posterior neck
    
    if any(x in name for x in ["flexion", "face up", "supine", "front"]):
        return "neck_flexion"  # anterior neck
    
    if any(x in name for x in ["sides", "lateral", "side"]):
        return "neck_lateral"
    
    if "isometric" in name and ("front" in name or "back" in name):
        return "neck_isometric_front_back"
    
    if "neck" in muscle.lower() or "cervical" in muscle.lower():
        return "neck_general"
    
    return None

In [81]:
def get_muscle_map(name):
    """
    Returns a list of (muscle_region, confidence) tuples for a given exercise name.
    Confidence: 0.0–1.0 (higher = stronger/primary target)
    """
    name = name.lower().strip()
    results = []

    # ───────────────────────────────────────────────
    # GLUTES (max / med / min)
    # ───────────────────────────────────────────────
    glute_keywords_high = ["hip thrust", "thrust", "glute bridge", "bridge", "frog pump", "kas glute bridge", "kas bridge"]
    if any(k in name for k in glute_keywords_high):
        results.append(("glute_maximus", 0.95))

    if "squat" in name or "goblet squat" in name:
        results.append(("glute_maximus", 0.70))
    if any(x in name for x in ["lunge", "split squat", "bulgarian split", "reverse lunge", "walking lunge"]):
        results.append(("glute_maximus", 0.65))
    if any(x in name for x in ["deadlift", "sumo deadlift", "romanian deadlift", "rack pull"]):
        results.append(("glute_maximus", 0.75))
    if any(x in name for x in ["kickback", "glute kickback", "donkey kick", "cable kickback", "hip extension"]):
        results.append(("glute_maximus", 0.90))
    if any(x in name for x in ["pull through", "cable pull through", "kettlebell swing"]):
        results.append(("glute_maximus", 0.80))

    if any(x in name for x in ["abduction", "hip abduction", "band abduction", "side abduction", "band walk", "monster walk", "clamshell", "fire hydrant", "lateral band"]):
        results.append(("glute_medius", 0.92))
        results.append(("glute_minimus", 0.78))
        results.append(("legs_abductors", 0.85))

    if any(x in name for x in ["single leg", "single-leg", "unilateral", "pistol squat", "single leg bridge", "hip hike"]):
        results.append(("glute_minimus", 0.75))
        results.append(("glute_medius", 0.70))

    # ───────────────────────────────────────────────
    # LEGS – Quads / Hamstrings / Hip Flexors / Abductors
    # ───────────────────────────────────────────────
    if any(x in name for x in ["squat", "front squat", "hack squat", "leg press", "goblet squat", "sissy squat"]):
        results.append(("legs_quads", 0.88))
    if any(x in name for x in ["lunge", "split squat", "bulgarian", "step up", "leg extension", "wall sit"]):
        results.append(("legs_quads", 0.82))

    if any(x in name for x in ["deadlift", "romanian deadlift", "stiff leg deadlift", "good morning", "leg curl", "lying leg curl", "seated leg curl", "nordic curl", "glute ham raise"]):
        results.append(("legs_hamstrings", 0.88))

    if any(x in name for x in ["leg raise", "hanging leg raise", "captain's chair", "knee raise", "mountain climber", "high knee", "flutter kick", "scissor kick"]):
        results.append(("legs_hip_flexors", 0.88))
        results.append(("rectus_abdominis", 0.65))  # secondary

    # ───────────────────────────────────────────────
    # ABS / CORE
    # ───────────────────────────────────────────────
    if any(x in name for x in ["crunch", "sit up", "sit-up", "v-up", "toe touch", "bicycle crunch", "ab crunch", "cable crunch"]):
        results.append(("rectus_abdominis", 0.95))
    if any(x in name for x in ["reverse crunch", "leg raise", "hanging leg raise", "flutter", "scissor"]):
        results.append(("rectus_abdominis", 0.88))  # lower emphasis

    if any(x in name for x in ["plank", "side plank", "forearm plank", "ab rollout", "ab wheel", "roll out", "dead bug", "bird dog"]):
        results.append(("transverse_core", 0.92))

    if any(x in name for x in ["russian twist", "woodchopper", "pallof press", "side bend", "oblique crunch", "windshield wiper"]):
        results.append(("obliques", 0.90))

    if "dragon flag" in name:
        results.append(("rectus_abdominis", 0.95))
        results.append(("transverse_core", 0.85))

    # ───────────────────────────────────────────────
    # CHEST
    # ───────────────────────────────────────────────
    if any(x in name for x in ["bench press", "chest press", "push up", "push-up", "dip", "chest dip", "close grip bench"]):
        results.append(("chest_pectorals", 0.92))
    if "incline" in name:
        results.append(("chest_pectorals", 0.88))  # upper
    if "decline" in name:
        results.append(("chest_pectorals", 0.85))  # lower
    if any(x in name for x in ["fly", "pec fly", "cable fly", "pec deck", "chest fly", "crossover"]):
        results.append(("chest_pectorals", 0.88))

    # ───────────────────────────────────────────────
    # BACK
    # ───────────────────────────────────────────────
    if any(x in name for x in ["pull up", "pull-up", "chin up", "chin-up", "lat pulldown", "pulldown", "wide grip"]):
        results.append(("back_lats", 0.92))
    if "pullover" in name or "straight arm pulldown" in name:
        results.append(("back_lats", 0.85))

    if any(x in name for x in ["row", "bent over row", "barbell row", "dumbbell row", "seated row", "cable row", "t-bar row", "inverted row"]):
        results.append(("back_rhomboids", 0.88))

    if any(x in name for x in ["shrug", "trap shrug", "upright row", "farmer walk", "face pull"]):
        results.append(("back_traps", 0.88))

    if any(x in name for x in ["back extension", "hyperextension", "superman"]):
        results.append(("back_lower_erectors", 0.82))

    # ───────────────────────────────────────────────
    # SHOULDERS
    # ───────────────────────────────────────────────
    if any(x in name for x in ["shoulder press", "military press", "overhead press", "arnold press", "seated press", "clean and press", "push press"]):
        results.append(("shoulders_anterior", 0.92))

    if any(x in name for x in ["lateral raise", "side lateral raise", "cable lateral raise", "upright row"]):
        results.append(("shoulders_lateral", 0.90))

    if any(x in name for x in ["rear delt", "rear deltoid fly", "face pull", "band pull apart", "reverse fly", "thread the needle"]):
        results.append(("shoulders_rear", 0.88))

    if any(x in name for x in ["internal rotation", "external rotation", "rotator cuff"]):
        results.append(("shoulders_rotator_cuff", 0.92))

    # ───────────────────────────────────────────────
    # ARMS
    # ───────────────────────────────────────────────
    if any(x in name for x in ["curl", "bicep curl", "barbell curl", "dumbbell curl", "hammer curl", "preacher curl", "concentration curl", "cable curl"]):
        results.append(("arms_biceps", 0.95))

    if any(x in name for x in ["tricep", "triceps extension", "overhead extension", "pushdown", "tricep pushdown", "skull crusher", "french press", "diamond push up", "close grip bench"]):
        results.append(("arms_triceps", 0.92))

    if any(x in name for x in ["wrist curl", "reverse wrist curl", "wrist roller", "farmer carry", "grip"]):
        results.append(("arms_forearms", 0.85))

    # ───────────────────────────────────────────────
    # CALVES
    # ───────────────────────────────────────────────
    if any(x in name for x in ["calf raise", "standing calf raise", "donkey calf raise", "smith calf raise"]):
        results.append(("calves_gastrocnemius", 0.92))
    if "seated calf raise" in name:
        results.append(("calves_soleus", 0.92))
    if "jump rope" in name or "heel drop" in name:
        results.append(("calves_gastrocnemius", 0.70))

    # ───────────────────────────────────────────────
    # NECK
    # ───────────────────────────────────────────────
    if "neck" in name or "head harness" in name:
        if any(x in name for x in ["extension", "face down", "prone"]):
            results.append(("neck_extension", 0.95))
        if any(x in name for x in ["flexion", "face up", "supine"]):
            results.append(("neck_flexion", 0.95))
        if any(x in name for x in ["side", "lateral", "sides"]):
            results.append(("neck_lateral", 0.92))
        else:
            results.append(("neck_general", 0.85))

    # ───────────────────────────────────────────────
    # FULL BODY / PLYOMETRIC / OTHER
    # ───────────────────────────────────────────────
    if any(x in name for x in ["burpee", "thruster", "snatch", "clean and jerk"]):
        results.append(("shoulders_anterior", 0.70))
        results.append(("legs_quads", 0.65))
        results.append(("glute_maximus", 0.60))

    # Sort by descending confidence
    results.sort(key=lambda x: x[1], reverse=True)
    return results


# ───────────────────────────────────────────────
# Test block (optional – remove if integrating)
# ───────────────────────────────────────────────
if __name__ == "__main__":
    test_exercises = [
        "Barbell Hip Thrust", "Bulgarian Split Squat", "Sumo Deadlift",
        "Nordic Hamstring Curl", "Ab Wheel Rollout", "Pallof Press",
        "Arnold Press", "Cable Rear Delt Fly", "Preacher Curl",
        "Seated Calf Raise", "Burpee", "Kettlebell Swing"
    ]
    for ex in test_exercises:
        print(f"{ex:35} → {get_muscle_map(ex)}")

Barbell Hip Thrust                  → [('glute_maximus', 0.95)]
Bulgarian Split Squat               → [('legs_quads', 0.88), ('legs_quads', 0.82), ('glute_maximus', 0.7), ('glute_maximus', 0.65)]
Sumo Deadlift                       → [('legs_hamstrings', 0.88), ('glute_maximus', 0.75)]
Nordic Hamstring Curl               → [('arms_biceps', 0.95)]
Ab Wheel Rollout                    → [('transverse_core', 0.92)]
Pallof Press                        → [('obliques', 0.9)]
Arnold Press                        → [('shoulders_anterior', 0.92)]
Cable Rear Delt Fly                 → [('chest_pectorals', 0.88), ('shoulders_rear', 0.88)]
Preacher Curl                       → [('arms_biceps', 0.95)]
Seated Calf Raise                   → [('calves_gastrocnemius', 0.92), ('calves_soleus', 0.92)]
Burpee                              → [('shoulders_anterior', 0.7), ('legs_quads', 0.65), ('glute_maximus', 0.6)]
Kettlebell Swing                    → [('glute_maximus', 0.8)]


In [82]:
rows = []

for _, row in df.iterrows():
    mappings = get_muscle_map(row["name"])

    for muscle, weight in mappings:
        rows.append({
            "exercise": row["name"],
            "muscle_region": muscle,
            "weight": weight,
            "rating": row["rating"]
        })

df_expanded = pd.DataFrame(rows)

In [83]:
df_expanded.head(20)

,exercise,muscle_region,weight,rating
0,partner plank band row,transverse_core,0.92,0.0
1,partner plank band row,back_rhomboids,0.88,0.0
2,banded crunch isometric hold,rectus_abdominis,0.95,5.9
3,fyr banded plank jack,transverse_core,0.92,5.9
4,banded crunch,rectus_abdominis,0.95,5.9
5,crunch,rectus_abdominis,0.95,5.9
6,decline band press sit-up,rectus_abdominis,0.95,5.9
7,decline band press sit-up,chest_pectorals,0.85,5.9
8,fyr2 banded frog pump,glute_maximus,0.95,5.9
9,barbell ab rollout - on knees,transverse_core,0.92,8.9


In [84]:
def classify_muscle_region(row):
    name = row["name"]
    muscle = row["muscle"]

    # 1. Glutes
    result = classify_glutes(name, muscle)
    if result:
        return result

    # 2. Abs
    result = classify_abs(name, muscle)
    if result:
        return result
    result = classify_shoulders(name, muscle)
    if result:
            return result
    result = classify_neck(name, muscle)
    if result:
            return result
    # 3. Calves
    result = classify_calves(name, muscle)
    if result:
        return result

    # 4. Add others (arms, legs, back)
    result = classify_arms(name, muscle)
    if result:
        return result

    result = classify_legs(name, muscle)
    if result:
        return result
    # chest
    result = classify_chest(name, muscle)
    if result:
        return result

    # adductors
    result = classify_adductors(name, muscle)
    if result:
        return result

    # plyometric
    result = classify_plyo(name)
    if result:
        return result

    # mobility
    result = classify_mobility(name)
    if result:
        return result

    result = classify_back(name, muscle)
    if result:
        return result

    result = classify_chest(name, muscle)
    if result:
        return result
    return "other"

In [76]:
df_expanded.isna().sum()

exercise         0
muscle_region    0
weight           0
rating           0
dtype: int64

In [85]:
len(df), len(df_expanded)

(2877, 3365)

In [107]:
import pandas as pd

# 1. Your muscle → image URL mapping (exactly as you gave)
# Note: glutes use list format → we take the first item
muscle_to_url = {
    # Abs
    "abs_general": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Abs.jpg?v=1601051622",
    "rectus_abdominis": "https://www.madscientistofmuscle.com/1-exercises/1-muscle-anatomy/graphics/rectus-abdominis.jpg",
    "transverse_core": "https://kinxlearning.com/cdn/shop/files/muscle-61_500x.jpg?v=1613518563",
    "obliques": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSdlZxloBeCPZiKsYR0i7I0imOG6J8bSyNiCg&s",

    # Legs
    "legs_general": "https://www.album-online.com/photos/prev/OTFlMzAxMA/album_alb3885948.jpg",
    "legs_hip_flexors": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRrnsZOCOmyZPhM3auBLWY7NtveLVp_11F6EA&s",
    "legs_adductors": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR0HeL_7E9c96i946uTekQh2SnAALvJ74NXAA&s",
    "legs_quads": "https://www.shutterstock.com/image-illustration/quadriceps-male-muscles-anatomy-muscle-260nw-489727177.jpg",
    "legs_hamstrings": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS_OWv94jxxO7g7rUVzmTJelOJFvjrh579tqA&s",

    # Chest & Neck
    "chest_pectorals": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQpss-AogAFbSeSywmcQersXZPsAzvEXHT2-Q&s",
    "chest_general": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Chest.jpg?v=1601050935",
    "neck_extension": "https://teachmeanatomy.info/neck/muscles/posterior-neck",
    "neck_lateral": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRC3LGjxddZZJvpVYGy-VV9jXcrvJ4KZFEz0g&s",
    "neck_general": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRC3LGjxddZZJvpVYGy-VV9jXcrvJ4KZFEz0g&s",

    # Arms
    "arms_biceps": "https://t3.ftcdn.net/jpg/05/04/89/18/240_F_504891829_C8C4uvSyiarKli1wYpnDmNvkqiWsE2KN.jpg",
    "arms_triceps": "https://barbend.com/wp-content/uploads/2019/04/Tricep-Muscle-Anatomy-1024x661.jpg",
    "arms_forearms": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Triceps.jpg?v=1601051412",
    "arms_general": "https://connect.healthkart.com/wp-content/uploads/2023/11/900-52-1.jpg",

    # Back
    "back_lats": "https://blog.warmbody-coldmind.com/wp-content/uploads/2024/05/training-barbell-lat-workout-latissimus-dorsi-anatomy.jpg",
    "back_rhomboids": "https://blog.infs.com/wp-content/uploads/2025/03/129458131_m.jpg",
    "back_general": "https://images.squarespace-cdn.com/content/v1/6853e9bd2f023c2ec20b8bf6/232997e8-7e70-40b5-8908-6fc556a8066d/BS+images+%284%29.png?format=2500w",

    # Glutes (using first URL from each list)
    "glute_maximus": "https://mdhealth.com.au/wp-content/uploads/2014/07/gluteus-maximus-1000x675.jpg",
    "glute_medius": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
    "glute_minimus": "https://cdn.shopify.com/s/files/1/0536/1244/5846/files/Gluteus_Minimus_Anatomy_QL_Claw_cdfdef14-251a-4865-a5c7-5a041fb53b22_300x300.jpg?v=1668039563",
}

# 2. Example: your DataFrame (replace with your real df)
df = pd.DataFrame({
    'muscle_region': [
        'glute_maximus', 'abs_general', 'shoulders_anterior', 'legs_general',
        'arms_biceps', 'rectus_abdominis', 'transverse_core', 'chest_pectorals',
        'neck_extension', 'back_rhomboids', 'legs_quads', 'legs_hamstrings',
        'obliques', 'glute_medius', 'arms_triceps', 'back_lats', 'legs_adductors',
        # ... add all your other regions here
    ],
    # you can add other columns like 'count': [...]
})

# 3. Add the new column 'image_url'
df['image_url'] = df['muscle_region'].map(muscle_to_url)

# Optional: fill missing URLs with a placeholder
df['image_url'] = df['image_url'].fillna('https://via.placeholder.com/600x400?text=No+Image+Found')

# 4. Quick checks
print("First 10 rows with new column:")
print(df.head(10))

print("\nHow many regions got an image URL?")
print(df['image_url'].notna().sum(), "out of", len(df))

print("\nRows without image:")
print(df[df['image_url'].str.contains("No Image", na=False)])

# Optional: save to file
# df.to_csv("muscles_with_images.csv", index=False)
# df.to_excel("muscles_with_images.xlsx", index=False)

First 10 rows with new column:
        muscle_region                                          image_url
0       glute_maximus  https://mdhealth.com.au/wp-content/uploads/201...
1         abs_general  https://cdn.shopify.com/s/files/1/1214/5580/fi...
2  shoulders_anterior  https://via.placeholder.com/600x400?text=No+Im...
3        legs_general  https://www.album-online.com/photos/prev/OTFlM...
4         arms_biceps  https://t3.ftcdn.net/jpg/05/04/89/18/240_F_504...
5    rectus_abdominis  https://www.madscientistofmuscle.com/1-exercis...
6     transverse_core  https://kinxlearning.com/cdn/shop/files/muscle...
7     chest_pectorals  https://encrypted-tbn0.gstatic.com/images?q=tb...
8      neck_extension  https://teachmeanatomy.info/neck/muscles/poste...
9      back_rhomboids  https://blog.infs.com/wp-content/uploads/2025/...

How many regions got an image URL?
17 out of 17

Rows without image:
Empty DataFrame
Columns: [muscle_region, image_url]
Index: []


In [117]:
import pandas as pd

# ───────────────────────────────────────────────
# 1. Load your file (change path/name as needed)
# ───────────────────────────────────────────────
df = pd.read_csv("ai_ready_exercises.csv")  # or pd.read_excel(...), etc.
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())

# ───────────────────────────────────────────────
# 2. Your exact muscle → image URL mapping (from your message)
#    Glutes use lists → we take the first item
# ───────────────────────────────────────────────
muscle_to_url = {
    # Abs
    "abs_general": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Abs.jpg?v=1601051622",
    "rectus_abdominis": "https://www.madscientistofmuscle.com/1-exercises/1-muscle-anatomy/graphics/rectus-abdominis.jpg",
    "transverse_core": "https://kinxlearning.com/cdn/shop/files/muscle-61_500x.jpg?v=1613518563",
    "obliques": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSdlZxloBeCPZiKsYR0i7I0imOG6J8bSyNiCg&s",

    # Legs
    "legs_general": "https://www.album-online.com/photos/prev/OTFlMzAxMA/album_alb3885948.jpg",
    "legs_hip_flexors": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRrnsZOCOmyZPhM3auBLWY7NtveLVp_11F6EA&s",
    "legs_adductors": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcR0HeL_7E9c96i946uTekQh2SnAALvJ74NXAA&s",
    "legs_quads": "https://www.shutterstock.com/image-illustration/quadriceps-male-muscles-anatomy-muscle-260nw-489727177.jpg",
    "legs_hamstrings": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS_OWv94jxxO7g7rUVzmTJelOJFvjrh579tqA&s",

    # Chest & Neck
    "chest_pectorals": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQpss-AogAFbSeSywmcQersXZPsAzvEXHT2-Q&s",
    "chest_general": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Chest.jpg?v=1601050935",
    "neck_extension": "https://teachmeanatomy.info/neck/muscles/posterior-neck",
    "neck_lateral": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRC3LGjxddZZJvpVYGy-VV9jXcrvJ4KZFEz0g&s",
    "neck_general": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRC3LGjxddZZJvpVYGy-VV9jXcrvJ4KZFEz0g&s",

    # Arms
    "arms_biceps": "https://t3.ftcdn.net/jpg/05/04/89/18/240_F_504891829_C8C4uvSyiarKli1wYpnDmNvkqiWsE2KN.jpg",
    "arms_triceps": "https://barbend.com/wp-content/uploads/2019/04/Tricep-Muscle-Anatomy-1024x661.jpg",
    "arms_forearms": "https://cdn.shopify.com/s/files/1/1214/5580/files/Muscle_Group_Triceps.jpg?v=1601051412",
    "arms_general": "https://connect.healthkart.com/wp-content/uploads/2023/11/900-52-1.jpg",

    # Back
    "back_lats": "https://blog.warmbody-coldmind.com/wp-content/uploads/2024/05/training-barbell-lat-workout-latissimus-dorsi-anatomy.jpg",
    "back_rhomboids": "https://blog.infs.com/wp-content/uploads/2025/03/129458131_m.jpg",
    "back_general": "https://images.squarespace-cdn.com/content/v1/6853e9bd2f023c2ec20b8bf6/232997e8-7e70-40b5-8908-6fc556a8066d/BS+images+%284%29.png?format=2500w",

    # Glutes (taking first URL from each list)
    "glute_maximus": "https://mdhealth.com.au/wp-content/uploads/2014/07/gluteus-maximus-1000x675.jpg",
    "glute_medius": "https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg",
    "glute_minimus": "https://cdn.shopify.com/s/files/1/0536/1244/5846/files/Gluteus_Minimus_Anatomy_QL_Claw_cdfdef14-251a-4865-a5c7-5a041fb53b22_300x300.jpg?v=1668039563",
}

# ───────────────────────────────────────────────
# 3. Add / standardize 'muscle_region' column
# ───────────────────────────────────────────────

# If 'muscle_region' does NOT exist yet, create it from your existing column
# CHANGE 'your_muscle_column_here' to the real name (e.g. 'target_muscle', 'body_part', 'muscle_group')
if 'muscle_region' not in df.columns:
    df['muscle_region'] = df['muscle'].fillna('Unknown').str.strip().str.lower()

# Optional: Normalize/clean names so they match dictionary keys better
df['muscle_region'] = df['muscle_region'].replace({
    'quads': 'legs_quads',
    'quadriceps': 'legs_quads',
    'hamstrings': 'legs_hamstrings',
    'hip flexors': 'legs_hip_flexors',
    'adductors': 'legs_adductors',
    'abs': 'abs_general',
    'rectus abdominis': 'rectus_abdominis',
    'transverse abdominis': 'transverse_core',
    'oblique': 'obliques',
    'pectoralis': 'chest_pectorals',
    'chest': 'chest_pectorals',
    'biceps': 'arms_biceps',
    'triceps': 'arms_triceps',
    'forearms': 'arms_forearms',
    'lats': 'back_lats',
    'rhomboids': 'back_rhomboids',
    'glute max': 'glute_maximus',
    'glute med': 'glute_medius',
    'glute min': 'glute_minimus',
    # add more mappings if needed
})

# ───────────────────────────────────────────────
# 4. Add the 'image_url' column
# ───────────────────────────────────────────────
df['image_url'] = df['muscle_region'].map(muscle_to_url)

# Fill missing with placeholder
df['image_url'] = df['image_url'].fillna('https://via.placeholder.com/600x400?text=No+Image+for+' + df['muscle_region'])

# ───────────────────────────────────────────────
# 5. Preview & save the updated file
# ───────────────────────────────────────────────
print("\nUpdated DataFrame preview (muscle_region + image_url):")
print(df[['muscle_region', 'image_url']].head(15))

print("\nSummary:")
print("Total rows:", len(df))
print("Rows with valid image URL:", df['image_url'].str.contains("http").sum())
print("Rows with placeholder:", df['image_url'].str.contains("No Image").sum())

# Save the enhanced file
output_file = "ai_ready_exercises_with_image_urls.csv"
df.to_csv(output_file, index=False)
print(f"\nSaved updated file: {output_file}")

Loaded shape: (2877, 8)
Columns: ['name', 'description', 'category', 'muscle', 'equipment', 'difficulty', 'rating', 'rating_description']

Updated DataFrame preview (muscle_region + image_url):
   muscle_region                                          image_url
0           core  https://via.placeholder.com/600x400?text=No+Im...
1           core  https://via.placeholder.com/600x400?text=No+Im...
2           core  https://via.placeholder.com/600x400?text=No+Im...
3           core  https://via.placeholder.com/600x400?text=No+Im...
4           core  https://via.placeholder.com/600x400?text=No+Im...
5           core  https://via.placeholder.com/600x400?text=No+Im...
6           core  https://via.placeholder.com/600x400?text=No+Im...
7           core  https://via.placeholder.com/600x400?text=No+Im...
8           core  https://via.placeholder.com/600x400?text=No+Im...
9           core  https://via.placeholder.com/600x400?text=No+Im...
10          core  https://via.placeholder.com/600x400?text

In [94]:
len(glute_image_urls['glute_maximus'])

3

In [97]:
glute_image_urls         

{'glute_maximus': ['https://mdhealth.com.au/wp-content/uploads/2014/07/gluteus-maximus-1000x675.jpg'],
 'glute_medius': ['https://nakoafit.com/wp-content/uploads/2023/10/gluteus-medius.jpg'],
 'glute_minimus': ['https://cdn.shopify.com/s/files/1/0536/1244/5846/files/Gluteus_Minimus_Anatomy_QL_Claw_cdfdef14-251a-4865-a5c7-5a041fb53b22_300x300.jpg?v=1668039563']}

In [113]:
df["image_url"] = df.get("image_url", None)
df["muscle_region"] = df.get("muscle_region", None)
df["name"] = df.get("name", None)
df["muscle"] = df.get("muscle", None)
df["description"] = df.get("description", None)
df["rating"] = df.get("rating", None)
df["difficulty"] = df.get("difficulty", None)
df["rating_description"] = df.get("rating_description", None)



In [105]:
df[df["muscle_region"].str.contains("core", case=False, na=False)]["muscle_region"].value_counts()

muscle_region
transverse_core    152
Name: count, dtype: int64

In [114]:
df.to_csv("ai_ex.csv", index=False)

In [112]:
print(df.columns)


Index(['muscle_region', 'image_url'], dtype='object')


In [62]:
df["muscle_region"] = df.apply(classify_muscle_region, axis=1)

In [63]:
df["muscle_region"].value_counts()

muscle_region
glute_maximus             558
abs_general               281
shoulders_anterior        271
legs_general              223
arms_biceps               186
rectus_abdominis          173
shoulders_general         156
transverse_core           152
chest_pectorals           109
neck_extension            106
back_rhomboids             78
arms_general               53
back_general               52
obliques                   44
glute_minimus              42
glute_general              42
arms_triceps               34
back_lats                  34
shoulders_lateral          33
neck_lateral               30
chest_general              29
glute_medius               25
shoulders_rear             25
arms_forearms              22
calves_gastrocnemius       21
chest_upper                17
calves_general             17
back_traps                 13
neck_flexion               11
mobility                    8
legs_hip_flexors            6
shoulders_rotator_cuff      4
back_lower_erectors       

In [68]:
df[df["muscle_region"] == "other"]["name"].head()

Series([], Name: name, dtype: object)

In [108]:
df.to_csv("ai_exercises.csv", index=False)